# 00 — Sanity Check (Colab)

**Goal:** load SmolVLM-Instruct, verify the pre-decoder extraction contract, and run one baseline question.

**Context:** This thesis prunes visual tokens *between the connector and the LLM decoder*. The wrapper below exposes exactly those points. We verify the verified architecture facts:

- `vision_model` = SigLIP ViT-SO400M (27×27 = 729 patches at 384px)
- `connector` = pixel-shuffle (scale 3) → **81 tokens** enter the LLM decoder
- `text_model` = SmolLM2 decoder (24 layers, hidden 2048)

> ⚠️ **Known correction:** our earlier notes said 729 tokens enter the LLM. The pixel-shuffle connector compresses to **81** for a 384×384 image (verified from `HuggingFaceTB/SmolVLM-Instruct/config.json`). This *strengthens* the redundancy argument — the model already works with 81 tokens, and we show you can prune ~50% of those and still get correct answers.

In [ ]:
!pip -q install torch transformers accelerate Pillow

import sys, os, torch
if not os.path.isdir("src"):
    # Running from the repo root in Colab
    print("Run this notebook from the repo root: text-guided-token-pruning/")

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from tgtp.models.smolvlm_wrapper import SmolVLMPruningWrapper, PRUNING_LOCATION

MODEL = "HuggingFaceTB/SmolVLM-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(MODEL, torch_dtype=dtype, trust_remote_code=True).to(device)
model.eval()
wrapper = SmolVLMPruningWrapper(model)
print("pruning location:", PRUNING_LOCATION)
print(wrapper.sanity_check_attributes())

In [ ]:
# Architecture verification (pre-decoder extraction points)
from PIL import Image
probe = Image.new("RGB", (384, 384))  # processors expect PIL images, not raw tensors
pv = processor(images=probe, return_tensors="pt")["pixel_values"].to(device)
with torch.no_grad():
    visual = wrapper.extract_visual_tokens(pv)
print("Visual tokens entering the LLM decoder:", visual.shape)  # expect (1, 81, 2048)

In [ ]:
# One baseline Q&A on a sample image (download a stock photo)
import requests
from PIL import Image
from io import BytesIO

url = "https://images.unsplash.com/photo-1502877338535-766e1452684a?w=600"  # a car
img = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
img.thumbnail((384, 384))
img

In [ ]:
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "What color is the car?"}]}]
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[img], return_tensors="pt").to(device)
out = model.generate(**inputs, max_new_tokens=20)
print(processor.decode(out[0], skip_special_tokens=True))

✅ **Sanity check passed** if the model answers sensibly.

Next: `01_token_visualization.ipynb` (heatmaps of relevance scores) and `03_random_vs_smart_pruning.ipynb` (the replication of our key finding).